In [ ]:
# Customer Churn Prediction for a Telecom Company

# Phase 11 – Model Interpretation

# Objective

# Interpret the best-performing machine learning model to understand how different features contribute to customer churn prediction.

# Tasks

# - Load the best model
# - Analyze Feature Importance
# - Compute Permutation Importance
# - Generate SHAP Summary Plot
# - Generate SHAP Waterfall Plot
# - Generate SHAP Bar Plot
# - Save all interpretation graphs

In [ ]:
# -----------------------------
# Phase 11 - Model Interpretation
# -----------------------------


# -----------------------------
# Import Libraries
# -----------------------------
import os
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.inspection import permutation_importance

# -----------------------------
# Create output folders
# -----------------------------
SAVE_PATH = "../images/09_Model_Interpretation"
REPORT_PATH = "../reports"
MODEL_PATH = "../models"

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

# -----------------------------
# Load dataset
# -----------------------------
X_train = joblib.load("../data/processed/X_train.pkl")
X_test = joblib.load("../data/processed/X_test.pkl")
y_test = joblib.load("../data/processed/y_test.pkl")

# -----------------------------
# Load best model
# -----------------------------
# Change this file if another model performed best.
model = joblib.load("../models/random_forest_tuned.pkl")

# -----------------------------
# Feature Importance
# -----------------------------
if hasattr(model, "feature_importances_"):
    importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance", ascending=False)
else:
    importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": np.zeros(X_train.shape[1])
    })

importance.to_excel("../reports/feature_importance.xlsx", index=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance.head(15), x="Importance", y="Feature")
plt.title("Top 15 Important Features")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "Feature_Importance.png"), dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# Permutation Importance
# -----------------------------
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

perm_df.to_excel("../reports/permutation_importance.xlsx", index=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=perm_df.head(15), x="Importance", y="Feature")
plt.title("Permutation Feature Importance")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "Permutation_Importance.png"), dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# SHAP Explainer
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Handle binary-class SHAP output
if isinstance(shap_values, list):
    shap_values_to_use = shap_values[1]
    base_value = explainer.expected_value[1]
else:
    shap_values_to_use = shap_values
    base_value = explainer.expected_value

# -----------------------------
# SHAP Summary Plot
# -----------------------------
shap.summary_plot(
    shap_values_to_use,
    X_test,
    show=False
)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "SHAP_Summary.png"), dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# SHAP Bar Plot
# -----------------------------
shap.summary_plot(
    shap_values_to_use,
    X_test,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "SHAP_Bar.png"), dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# SHAP Waterfall Plot
# -----------------------------
sample_idx = 0

waterfall_explanation = shap.Explanation(
    values=shap_values_to_use[sample_idx],
    base_values=base_value,
    data=X_test.iloc[sample_idx],
    feature_names=X_test.columns
)

shap.plots.waterfall(waterfall_explanation, show=False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "SHAP_Waterfall.png"), dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# Save SHAP values
# -----------------------------
joblib.dump(shap_values, "../models/shap_values.pkl")

# -----------------------------
# Top 10 features table
# -----------------------------
top10 = importance.head(10)
top10.to_excel("../reports/top10_features.xlsx", index=False)
top10

In [ ]:
# Observations

# The interpretation techniques identified the most influential features affecting customer churn.

# The feature importance and SHAP analysis help explain model predictions and provide transparency for business stakeholders.

# These insights can guide telecom companies in developing targeted customer retention strategies.